# ComfyUI on Kaggle

这是一份全新 notebook（不修改你原有文件），用于在 Kaggle 里运行 ComfyUI。

按顺序执行下面代码单元：
1. 克隆 ComfyUI
2. 安装 ComfyUI Manager
3. 安装缺失的自定义节点
4. 安装依赖
5. 启动 ComfyUI + Ngrok
6. 故障排查：查看日志与端口
7. 保活（可选）

注意：Kaggle Session 重启后需要重新运行。

In [ ]:
# 1) 克隆 ComfyUI

%cd /kaggle/
!git clone https://github.com/comfyanonymous/ComfyUI.git

print("\nComfyUI 克隆完成")

In [ ]:
# 2) 克隆 ComfyUI-Manager

%cd /kaggle/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git

In [ ]:
# 3) 安装其他社区节点和依赖

import os, re, shutil, subprocess, sys, tempfile, venv

CUSTOM_NODES = "/kaggle/ComfyUI/custom_nodes"
VENV_DIR = "/kaggle/working/comfyui-venv"
RUNTIME_PYTHON = f"{VENV_DIR}/bin/python"

# Avoid interactive git prompts in Kaggle
os.environ["GIT_TERMINAL_PROMPT"] = "0"
subprocess.call(["git", "config", "--global", "credential.helper", ""])

# Run venv Python with a sanitized environment to avoid Kaggle sitecustomize issues
CLEAN_ENV = os.environ.copy()
CLEAN_ENV.pop("PYTHONPATH", None)
CLEAN_ENV.pop("PYTHONHOME", None)
CLEAN_ENV["PYTHONNOUSERSITE"] = "1"

if not os.path.exists(RUNTIME_PYTHON):
    print("[setup] Creating ComfyUI venv for node dependencies...")
    try:
        venv.create(VENV_DIR, with_pip=True)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        subprocess.check_call([sys.executable, "-m", "virtualenv", VENV_DIR])

# Bootstrap pip tooling and wrapt inside the runtime venv first
subprocess.check_call([RUNTIME_PYTHON, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel", "wrapt"], env=CLEAN_ENV)

repos = [
    ("ComfyUI-WanVideoWrapper",        "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),
    ("comfyui_controlnet_aux",         "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("ComfyUI-KJNodes",                "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-Easy-Use",               "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-VideoHelperSuite",       "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI_essentials",             "https://github.com/cubiq/ComfyUI_essentials.git"),
    ("audio-separation-nodes-comfyui", "https://github.com/christian-byrne/audio-separation-nodes-comfyui.git"),
    ("comfyui-various",                "https://github.com/jamesWalker55/comfyui-various.git"),
    ("CRT-Nodes",                      "https://github.com/crt-nodes/CRT-Nodes.git"),
]

def _owner_repo(url):
    m = re.search(r"github.com/([^/]+)/([^/.]+)(?:\.git)?$", url)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def install_repo(name, url, dest):
    if os.path.isdir(dest) and os.listdir(dest):
        print(f"[skip] {name} already exists")
        return True

    print(f"[clone] {name}")
    r = subprocess.run(["git", "clone", "--depth=1", url, dest], env=CLEAN_ENV)
    if r.returncode == 0:
        return True

    print(f"[warn] git clone failed for {name}, trying zip fallback")
    owner, repo = _owner_repo(url)
    if not owner:
        return False

    with tempfile.TemporaryDirectory() as td:
        zip_path = os.path.join(td, f"{repo}.zip")
        for branch in ("main", "master"):
            zip_url = f"https://codeload.github.com/{owner}/{repo}/zip/refs/heads/{branch}"
            d = subprocess.run(["wget", "-q", "-O", zip_path, zip_url], env=CLEAN_ENV)
            if d.returncode != 0 or (not os.path.exists(zip_path)) or os.path.getsize(zip_path) == 0:
                continue

            u = subprocess.run(["unzip", "-q", zip_path, "-d", td], env=CLEAN_ENV)
            if u.returncode != 0:
                continue

            extracted = os.path.join(td, f"{repo}-{branch}")
            if os.path.isdir(extracted):
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.move(extracted, dest)
                print(f"[ok] Installed {name} via zip fallback")
                return True

    return False

clone_failed = []
for name, url in repos:
    dest = os.path.join(CUSTOM_NODES, name)
    ok = install_repo(name, url, dest)
    if not ok:
        clone_failed.append(name)

pip_failed = []
for name, _ in repos:
    if name in clone_failed:
        continue
    req = os.path.join(CUSTOM_NODES, name, "requirements.txt")
    if os.path.exists(req):
        print(f"[pip] installing requirements for {name}")
        pr = subprocess.run([RUNTIME_PYTHON, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", req], env=CLEAN_ENV)
        if pr.returncode != 0:
            pip_failed.append(name)

if clone_failed or pip_failed:
    print("\n[summary] Some nodes were not fully prepared")
    if clone_failed:
        print(f"  clone failed: {clone_failed}")
    if pip_failed:
        print(f"  requirements failed: {pip_failed}")
    raise RuntimeError("Custom node installation incomplete. See summary above.")
else:
    print("\n[done] All custom nodes installed and dependencies prepared.")

In [ ]:
# 4) 设置 venv 并安装 ComfyUI 依赖

import os, sys, subprocess, venv

VENV_DIR = "/kaggle/working/comfyui-venv"
PYTHON = f"{VENV_DIR}/bin/python"

if not os.path.exists(PYTHON):
    print("Creating isolated venv for ComfyUI...")
    try:
        venv.create(VENV_DIR, with_pip=True)
    except Exception as e:
        print(f"venv failed: {e}. Falling back to virtualenv...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        subprocess.check_call([sys.executable, "-m", "virtualenv", VENV_DIR])

subprocess.check_call([PYTHON, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([PYTHON, "-m", "pip", "install", "-r", "/kaggle/ComfyUI/requirements.txt"])
print(f"ComfyUI Python: {PYTHON}")

In [ ]:
# 5) 设置 ngrok 隧道并启动 ComfyUI

import os, sys, subprocess, gc
from kaggle_secrets import UserSecretsClient

VENV_DIR = "/kaggle/working/comfyui-venv"
PYTHON = f"{VENV_DIR}/bin/python"

# Set NGROK_AUTHTOKEN in Kaggle Secrets. Fallback: paste token directly.
# 获取 NGROK_AUTHTOKEN
try:
    Ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception as e:
    print(f"Error getting NGROK_AUTHTOKEN: {e}")
    print("请在 Kaggle → Add-ons → Secrets 中添加 Label: NGROK_AUTHTOKEN, Value: 你的 ngrok token")
    raise
Ngrok_domain = ""  # optional, leave empty if you don't have a domain
port = 8188

# -----------------

required_paths = [
    "/kaggle/ComfyUI/main.py",
    PYTHON,
    "/kaggle/ComfyUI/models/checkpoints",
    "/kaggle/ComfyUI/models/vae",
    "/kaggle/ComfyUI/models/clip_vision",
    "/kaggle/ComfyUI/models/upscale_models",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"Not ready yet. Missing required paths: {missing}")

if not Ngrok_token:
    raise RuntimeError("Missing NGROK_AUTHTOKEN. Set it in Kaggle Secrets, then rerun.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "pyngrok==6.1.0"])

from pyngrok import ngrok, conf

gc.collect()

try:
    ngrok.set_auth_token(Ngrok_token)
    ngrok.kill()
    if Ngrok_domain:
        srv = ngrok.connect(port, domain=Ngrok_domain)
    else:
        srv = ngrok.connect(port)
    print(f"Ngrok Tunnel is active at: {srv.public_url}")

    # Start ComfyUI using the isolated Python environment.
    subprocess.check_call([PYTHON, "/kaggle/ComfyUI/main.py"])
except Exception as e:
    print(f"Error starting ngrok tunnel: {e}")

In [ ]:
# 6) 故障排查：查看日志与端口
!echo "===== ComfyUI 进程 ====="
!ps -ef | grep -i "comfyui\|main.py" | grep -v grep || true
!echo "===== 监听端口 (8188) ====="
!ss -lntp | grep 8188 || true
!echo "===== ngrok 隧道 ====="
!curl -s http://127.0.0.1:4040/api/tunnels | python -m json.tool || echo "ngrok not running or API not accessible"

In [ ]:
# 7) 保活（可选）
import time
import urllib.request

print(f"Keepalive started. Checking ComfyUI on port 8188 every 30s")
print("Press Interrupt to stop this cell.")

while True:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=5) as resp:
            print(time.strftime("%H:%M:%S"), "alive", "status=", resp.status)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "check_failed", str(e))
    time.sleep(30)

# Install Model

**Install Checkpoints**

In [ ]:
# Install a model to permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = "https://huggingface.co/adamo1139/stable-diffusion-3-medium-ungated/resolve/main/sd3_medium_incl_clips_t5xxlfp8.safetensors?download=true"
model_name = "sd3_medium_incl_clips_t5xxlfp8.safetensors"

%cd /kaggle/ComfyUI/models/checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

**Install Loras**

In [ ]:
# Install a model to permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = "https://civitai.com/api/download/models/347271?type=Model&format=SafeTensor"
model_name = "Picture_enhancer.safetensors"

%cd /kaggle/ComfyUI/models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

**Install VAE**

In [ ]:
# Install a model to permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = "https://civitai.com/api/download/models/309729?type=VAE&format=SafeTensor"
model_name = "sdxl_vae.safetensors"

%cd /kaggle/ComfyUI/models/vae
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

**Install CLIP Vision H** (clip_vision_h.safetensors)

In [ ]:
import os
model_url = "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors"
model_name = "clip_vision_h.safetensors"
model_path = f"/kaggle/ComfyUI/models/clip_vision/{model_name}"

if os.path.exists(model_path):
    print(f"[skip] {model_name} already exists")
else:
    %cd /kaggle/ComfyUI/models/clip_vision
    get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

**Install RealESRGAN x2** (RealESRGAN_x2.pth)

In [ ]:
import os
model_url = "https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth"
model_name = "RealESRGAN_x2.pth"
model_path = f"/kaggle/ComfyUI/models/upscale_models/{model_name}"

if os.path.exists(model_path):
    print(f"[skip] {model_name} already exists")
else:
    %cd /kaggle/ComfyUI/models/upscale_models
    get_ipython().system(f'wget -O "{model_name}" "{model_url}"')